# 02. EDA & Distributional Drift

Objetivo: Cálculo del Population Stability Index (PSI) para contrastar los 209 HCPs etiquetados vs los 15,000 no etiquetados. Aplicación de Isolation Forest espacial para señalar outliers agresivos sin eliminarlos.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def calculate_psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    """
    Calculates Population Stability Index (PSI) to detect distributional drift 
    between labeled sample and the unlabeled population.
    """
    epsilon = 1e-5
    
    if len(expected) == 0 or len(actual) == 0:
        return 0.0
        
    # Generate identical bins for both distributions
    breakpoints = np.linspace(0, max(np.max(expected), np.max(actual)), bins + 1)
    
    expected_pct = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_pct = np.histogram(actual, bins=breakpoints)[0] / len(actual)
    
    # Defensively add epsilon to avoid division by zero or log(0)
    expected_pct = np.where(expected_pct == 0, epsilon, expected_pct)
    actual_pct = np.where(actual_pct == 0, epsilon, actual_pct)
    
    psi_values = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)
    return np.sum(psi_values)


## Isolation Forest (Spatial Outlier Flagging)

In [ ]:
def flag_spatial_outliers(df: pd.DataFrame, features_to_evaluate: list) -> pd.DataFrame:
    """
    Implements a spatial Isolation Forest to strictly flag extreme outliers 
    without dropping any records.
    """
    logger.info("Running Isolation Forest to flag outliers.")
    df_out = df.copy()
    
    available_features = [f for f in features_to_evaluate if f in df_out.columns]
    
    if not available_features or df_out.empty:
        logger.warning("Not enough data/features to run Isolation Forest.")
        df_out['is_outlier'] = False
        return df_out

    iso_forest = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
    
    # -1 means outlier, 1 means inlier
    preds = iso_forest.fit_predict(df_out[available_features].fillna(0))
    
    df_out['is_outlier'] = (preds == -1)
    return df_out
